# 15.10 模型究竟有多大？


「這個MoE有一億個參數，但每個token只啟動兩千萬個」聽起來像兩種互相矛盾的大小。其實前者問模型要保存多少可學數字，後者問每個token經過哪些權重。即使某位expert這次沒有被選中，它的權重仍需要儲存，下次不同token可能會用到它。這和[Dense的共享](https://birdhackor.github.io/tiny-perceptron-vlm/15.1.html)是同一個區分：儲存了哪些規則，與這個token使用哪些規則，是兩個不同問題。少選expert不會刪掉未選者的權重。

用容易核對的尺寸計算。每個expert把寬度4擴成8，再收回4，兩張表共 `4×8+8×4=64` 個權重，不計bias。四位expert共256，router把四個輸入特徵轉成四個分數，需要 `4×4=16` 個。此MoE FFN總共272個權重。每個token若選兩位，經過的expert權重約128個，加router16，共144。這是此FFN的局部估計，還沒有加上attention、字嵌入與其他層。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
width, hidden, k = 4, 8, 2
one_expert = width * hidden + hidden * width
for experts in (4, 8):
    router = width * experts
    total = experts * one_expert + router
    active = k * one_expert + router
    print("expert數", experts, "總權重", total, "每token使用估計", active, "FP32權重bytes", total * 4)

程式只計數，不建立隨機模型或聲稱測到執行記憶體。`width`是每個token的特徵數，`hidden`是FFN中間寬度，`k`是每個token選幾位。四位時輸出272、144、1088；八位時544、160、2176。FP32每個權重用四bytes，所以最後一欄只算權重數字的儲存量，不含容器與額外狀態。

expert數加倍、k固定，全部expert的儲存增加一倍，啟動的expert部分仍為128；但router也要替更多候選打分，所以使用估計從144增為160，不是完全不變。若加入完整模型的共享部分，總權重和每token使用估計都要再加它們，不能把這兩個FFN數字當作整個模型大小。

「使用了多少權重」也不是精確乘加次數，更不是實測速度。運算形狀、分派開銷及記憶體搬移都會影響成本；訓練還要存[梯度](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.6)，也就是微小改權重對誤差的影響，以及優化器狀態：更新權重時用來記住過去變化的附加數字。推理會有[cache快取](https://birdhackor.github.io/tiny-perceptron-vlm/16.3.html)，為接續生成保存已算過的前文資訊，還有每次運算產生的中間特徵。因此報告MoE規模時，至少應列總參數、每token選幾位，以及這個估計涵蓋哪些模組。只說「活躍參數少」不足以讓小記憶體裝置容納全部權重。

練習只把k由2改成1。先預測總權重和FP32 bytes完全不變，四位時使用估計降為80、八位時降為96，再重跑核對。這直接區分「模型能存多少規則」與「這次計算花了多少規則」兩個問題。

<details>
<summary>選讀：來源、量測條件與原始實報</summary>

完整訓練模型的兩層共有264,704個expert參數、512個router參數與75,392個其他參數，總數340,608。報告將全部非expert參數，加上每層選中expert的參數，稱為`active_parameter_proxy_per_token`：top-1得到142,080，top-2得到208,256。這個容量代理連整張字表與位置表都算入。[字表](https://birdhackor.github.io/tiny-perceptron-vlm/2.1.html)按文字單位的編號（token ID）查出一列特徵，[位置表](https://birdhackor.github.io/tiny-perceptron-vlm/4.1.html)則按位置編號查出一列特徵；一個token通常各查一列，這個代理卻刻意把兩張完整表的參數全列入。因此它並非每個token實際讀取的權重格數，也不是浮點運算次數；例如一次浮點乘法加上一次加法，通常計為兩次運算。它適合明確記錄匹配口徑，不能單獨證明乘加量、速度或GPU峰值省了多少。

</details>
